In [0]:
%run /Workspace/Users/erica.roque@gmail.com/Databricks-Certified-Data-Engineer-Associate/Includes/Copy-Datasets

In [0]:
USE CATALOG demo_workspace;
USE SCHEMA book_store;

In [0]:
%python
files = dbutils.fs.ls(f"{dataset_bookstore}/customers-json")
display(files)

In [0]:
select count(*) from json.`/Workspace/Users/erica.roque@gmail.com/demo-datasets/bookstore/customers-json`





In [0]:
select *, _metadata.file_name, _metadata.file_path
from json.`/Workspace/Users/erica.roque@gmail.com/demo-datasets/bookstore/customers-json`;

In [0]:
SELECT * FROM text.`dbfs:/Workspace/Users/erica.roque@gmail.com/demo-datasets/bookstore/customers-json`

In [0]:
SELECT * FROM binaryFile.`dbfs:/Workspace/Users/erica.roque@gmail.com/demo-datasets/bookstore/customers-json`

In [0]:
select * from csv.`/Workspace/Users/erica.roque@gmail.com/demo-datasets/bookstore/books-csv`



In [0]:
CREATE OR REPLACE TEMP VIEW book_csv_temp
(book_id STRING, title STRING, author STRING, category STRING, price STRING)
USING csv
OPTIONS
(
  header = "true",
  delimiter = ";",
  path = "/Workspace/Users/erica.roque@gmail.com/demo-datasets/bookstore/books-csv"
)




In [0]:
SELECT * FROM book_csv_temp

In [0]:
CREATE SCHEMA IF NOT EXISTS demo_workspace.book_store

In [0]:

CREATE TABLE IF NOT EXISTS demo_workspace.book_store.books_csv AS 
SELECT * FROM book_csv_temp;


In [0]:
select * from demo_workspace.book_store.books_csv

In [0]:
describe extended demo_workspace.book_store.books_csv

In [0]:
describe history demo_workspace.book_store.books_csv

In [0]:
%python
files = dbutils.fs.ls(f"{dataset_bookstore}/books-csv")
display(files)

In [0]:
%python
(spark.read
    .table('demo_workspace.book_store.books_csv')
    .write
    .mode("append")
    .format("csv")
    .option('header', 'true')
    .option('delimiter', ';')
    .save(f"{dataset_bookstore}/books-csv")
    )

In [0]:
%python
files = dbutils.fs.ls(f"{dataset_bookstore}/books-csv")
display(files)

In [0]:
select count(*) from demo_workspace.book_store.books_csv

In [0]:
refresh table demo_workspace.book_store.books_csv

In [0]:
analyze table demo_workspace.book_store.books_csv compute statistics

In [0]:
select *, _metadata.file_name, _metadata.file_path
from json.`/Workspace/Users/erica.roque@gmail.com/demo-datasets/bookstore/customers-json`;

In [0]:
CREATE OR REPLACE TEMP VIEW customer_tmp AS 

select * from json.`/Workspace/Users/erica.roque@gmail.com/demo-datasets/bookstore/customers-json`



In [0]:
select * from customer_tmp

In [0]:

CREATE TABLE IF NOT EXISTS demo_workspace.book_store.customers AS 
SELECT * FROM customer_tmp;

In [0]:
describe extended demo_workspace.book_store.customers


In [0]:
CREATE OR REPLACE TEMP VIEW customer_tmp AS 

select * from json.`/Workspace/Users/erica.roque@gmail.com/demo-datasets/bookstore/customers-json`


In [0]:
create or replace temp view temp_books_unparsed 
(bookd_id String, title String, author String, category String, price double)
using csv
options(
 header = 'true',
 delimiter = ';',
 path = '/Workspace/Users/erica.roque@gmail.com/demo-datasets/bookstore/books-csv/export_*.csv'
);



Create table books as 
select * from temp_books_unparsed;

select * from books


In [0]:
describe extended books

In [0]:
select _metadata.file_name, _metadata.file_path, _metadata.file_size, _metadata.file_modification_time 
from demo_workspace.book_store.books

In [0]:
USE CATALOG demo_workspace;
USE SCHEMA book_store;

SELECT count(*) FROM json.`/Workspace/Users/erica.roque@gmail.com/demo-datasets/bookstore/customers-json`;


In [0]:
create table if not exists orders as 
select * from parquet.`/Workspace/Users/erica.roque@gmail.com/demo-datasets/bookstore/orders`

In [0]:
select count(*) from orders

In [0]:
describe history orders

In [0]:
insert overwrite orders
select * from parquet.`/Workspace/Users/erica.roque@gmail.com/demo-datasets/bookstore/orders`;
describe history orders

In [0]:
insert overwrite orders
select * from parquet.`/Workspace/Users/erica.roque@gmail.com/demo-datasets/bookstore/orders-new`;

In [0]:
select count(*) from orders version as of 2

In [0]:
select count(*) from customers
--1700

In [0]:
create or replace temp view customers_update as
select * from json.`/Workspace/Users/erica.roque@gmail.com/demo-datasets/bookstore/customers-json-new`;

Merge into customers c
using customers_update u
on c.customer_id = u.customer_id
when matched and c.email is null and u.email is not null then 
update set c.email = u.email , c.updated = u.updated

when not matched then insert *

In [0]:
describe history customers

In [0]:
create or replace temp view books_update 
(book_id String, title String, author String, category String, price double)
using csv
options(
    path = '/Workspace/Users/erica.roque@gmail.com/demo-datasets/bookstore/books-csv-new',
    delimiter = ';',
    header = 'true'
);
select * from books_update

In [0]:
Merge into books b
using books_update u
on b.book_id = u.book_id and b.title = u.title
when not matched and u.category = 'Computer Science' then
Insert *

In [0]:
ALTER TABLE demo_workspace.book_store.books 
SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name');

In [0]:
ALTER TABLE demo_workspace.book_store.books RENAME COLUMN bookd_id TO book_id;

In [0]:
select * from books

In [0]:
show TBLPROPERTIES demo_workspace.book_store.books;


In [0]:
select customer_id, profile:first_name as firstname, profile:last_name as lastname
from customers

In [0]:
select profile
from customers
limit 1

In [0]:
create or replace temp view parsed_customers as
select customer_id, from_json(profile, schema_of_json('{"first_name":"Susana","last_name":"Gonnely","gender":"Female","address":{"street":"760 Express Court","city":"Obrenovac","country":"Serbia"}}')) as profile_struct
from customers;

select * from parsed_customers


In [0]:
create or replace temp view customers_final as
Select customer_id, profile_struct.*
from parsed_customers;

select * from customers_final

In [0]:

select order_id, customer_id, explode(books) as book
from orders

In [0]:
select customer_id,
collect_set(order_id) as orders_set,
collect_set(books.book_id) as books_set
from orders
group by customer_id

In [0]:
create or replace view orders_enriched as 
select * 
from (
    select *, explode(books) as book
    from orders) o
    inner join books b
    on b.book_id = o.book.book_id ;
select * from orders_enriched

In [0]:
create or replace temp view order_update AS
select * from parquet.`/Workspace/Users/erica.roque@gmail.com/demo-datasets/bookstore/orders-new`;

select * from orders o
minus
select * from order_update


In [0]:
create or replace table transaction as
Select * from 
(
    Select customer_id, book.book_id as book_id, book.quantity as quantity
from orders_enriched
) PIVOT(
 sum(quantity) for book_id in (
 'B01','B02','B03','B04','B05','B06','B07','B08','B09','B10','B11','B12'
 )
);

 SELECT * FROM transaction





